# Polis: using graph for community detection

version 0.0

## requirements
red-dwarf requirements + leidenlalg library
 

In [4]:
%pip install --quiet red-dwarf[all]@git+https://github.com/polis-community/red-dwarf.git@main
%pip install --quiet leidenalg

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## loading the conversation and building the opinion matrix

In [5]:
from reddwarf.data_loader import Loader
from reddwarf.utils.statements import process_statements

# You can also load from a Polis URL: Loader(polis_id="r2dfw8eambusb8buvecjt")
loader = Loader(filepaths=[
    "../../tests/fixtures/above-100-ptpts/votes.json",
    "../../tests/fixtures/above-100-ptpts/comments.json",
    "../../tests/fixtures/above-100-ptpts/conversation.json",
])

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(
    statement_data=loader.comments_data
)

# Build a lookup for statement text
stmt_text = {}
for c in loader.comments_data:
    sid = c.get("tid", c.get("statement_id"))
    stmt_text[sid] = c.get("txt", "")

print(f"Loaded {len(loader.votes_data)} votes")

print(f"Statements: {len(loader.comments_data)} total, {len(mod_out_statement_ids)} moderated out")


# nb of participants
list_of_ids_of_answer = [vote_data['participant_id'] for vote_data in loader.votes_data]
unique_id = np.unique(list_of_ids_of_answer)
nb_participants = len(unique_id)



Loaded 5216 votes
Statements: 63 total, 7 moderated out


In [6]:
conversation = pd.DataFrame(loader.votes_data)
print(conversation.head())
opinion_matrix = pd.crosstab(conversation['participant_id'], conversation['statement_id'], conversation['vote'], aggfunc='sum', dropna=False)

print(opinion_matrix.head())

# how many values in the conversation matrix are nan
# print(opinion_matrix.isna().sum().sum()/opinion_matrix.size)



   participant_id  statement_id  vote  weight_x_32767      modified  \
0               0             1     1               0  1.515621e+12   
1               0             2     1               0  1.515621e+12   
2               0             3     1               0  1.515621e+12   
3               0             4     1               0  1.515621e+12   
4               0             5    -1               0  1.515621e+12   

  conversation_id  
0      3ntrtcehas  
1      3ntrtcehas  
2      3ntrtcehas  
3      3ntrtcehas  
4      3ntrtcehas  
statement_id     1    2    3    4    5    6    7    8    9    10  ...  52  54  \
participant_id                                                    ...           
0               1.0  1.0  1.0  1.0 -1.0  1.0  0.0  1.0  NaN  NaN  ... NaN NaN   
1               1.0  1.0  1.0  1.0  1.0 -1.0  1.0  0.0  1.0 -1.0  ... NaN NaN   
2               1.0 -1.0  1.0 -1.0  1.0  1.0 -1.0  1.0  NaN  NaN  ... NaN NaN   
3               0.0  1.0  1.0 -1.0  1.0 -1.0 -1.

## Building the graph of opinions

"find_adjacency_matrix" takes as imput an opinion matrix and outputs the adjacency matrix of the participants' graph.

the "participants' graph" is a weighted undirected graph where the nodes are the participants.  Each participant is linked to each other with an edge whose weight is given by:

weight_{i,j} = nb_common_answers_{i, j} - nb_opposing_answers_{i, j}. 

- "nb_common_answers_{i,j}" is the number of statements where participant i and participant j voted both agree, disagree or unsure.
-  "nb_opposing_answers_{i, j}" is the number of statements where participant i voted agree and participant j voted disagree or the other way around.

Variants of the participants' should be tested. 

In [7]:

import pandas as pd
import numpy as np

def find_adjacency_matrix(opinion_matrix: pd.DataFrame) -> pd.DataFrame:
    A = opinion_matrix.to_numpy()
    # matches is a (A.shape[0] A.shape[0] A.shape[1]) numpy array where matches[i, j, k] is True if participant i and participant j gave the same vote for statement k
    matches = A[:, None, :] == A[None, :, :]

    different = (
        (A[:, None, :] != A[None, :, :]) &
        pd.notna(A[:, None, :]) &
        pd.notna(A[None, :, :])
    )

    similarity_np = matches.sum(axis=2) -different.sum(axis=2)
    return pd.DataFrame(
        similarity_np,
        index=opinion_matrix.index,
        columns=opinion_matrix.index
    )

adjacency_matrix  = find_adjacency_matrix(opinion_matrix)

print(adjacency_matrix.head())

idx = np.arange(len(adjacency_matrix))

for i in idx: 
    adjacency_matrix.iloc[i, i] = 0

    
print(adjacency_matrix.head())

participant_id  0    1    2    3    4    5    6    7    8    9    ...  257  \
participant_id                                                    ...        
0                13    2    0    2    7    2    0    4    3    2  ...   -3   
1                 2   25   -1    5    5   -1   -1    7    5    2  ...    2   
2                 0   -1    9    3   -1   -4    0    1    3    4  ...    0   
3                 2    5    3   47   15    1   -1   10    3    0  ...    2   
4                 7    5   -1   15   43    7   -1   18    7    0  ...    0   

participant_id  258  259  260  261  262  263  264  265  266  
participant_id                                               
0                -4    0   -2    2   -1    0   -3    0   -1  
1                -4    3    0   -1   -4   -1   -3    2   -1  
2                 1   -1   -1    1    0   -1   -2   -1    0  
3               -12    1    6   -1   -3   -1   -3    2   -1  
4                -8    1    4   -1   -3    1   -3    2   -1  

[5 rows x 267 colu

## Find communities with leiden algorithm

[TO DO: write a more precise explanation]

This part find community.  We build two graphs from the weighted graphs: one using the edges with positive weight and the other with edges with negative weight.  The optimiser find partitions that maximises the modularity in the positive graph and minimises the modularity in the negative graph

link to a video explaining modularity: https://www.youtube.com/watch?v=Xt0vBtBY2BU


we use the leiden algorithm, whch is a variant of the louvan method: https://pypi.org/project/leidenalg/

# 

In [ ]:
# Split the signed adjacency matrix into two non-negative matrices,
# since modularity-based Leiden requires non-negative edge weights.
# Invariant: adjacency_matrix = A_pos - A_neg
A_pos = np.maximum(adjacency_matrix.to_numpy(), 0)   # keep positive edges, zero out negatives
A_neg = np.maximum(-adjacency_matrix.to_numpy(), 0)  # keep magnitudes of negative edges, zero out positives

# Build two undirected weighted graphs over the same set of vertices:
# one for positive relationships, one for negative relationships.
# Edge weights are stored in the "weight" attribute.
G_pos = ig.Graph.Weighted_Adjacency(
    A_pos.tolist(),
    mode="undirected",
    attr="weight"
)

G_neg = ig.Graph.Weighted_Adjacency(
    A_neg.tolist(),
    mode="undirected",
    attr="weight"
)

# Create modularity partition objects, one per graph.
# Each tracks a community assignment and computes modularity
# using the "weight" edge attribute.
part_pos = la.ModularityVertexPartition(
    G_pos,
    weights="weight"
)

part_neg = la.ModularityVertexPartition(
    G_neg,
    weights="weight"
)

# Signed Leiden optimization (multiplex):

pos_weight = 1.0  # weight of positive layer in multiplex optimization
neg_weight = 1.0  # weight of negative layer in multiplex optimization

# Find ONE community assignment shared across both layers that maximizes
#     1 * Q_pos  -  1 * Q_neg
# - Q_pos is maximized: positive edges pull nodes into the same community.
# - Q_neg is minimized (negative layer weight): negative edges push
#   nodes into different communities.
optimiser = la.Optimiser()

optimiser.optimise_partition_multiplex(
    [part_pos, part_neg],
    layer_weights=[pos_weight, neg_weight]
)

# Final community label for each node (index i -> community ID of node i).
membership = part_pos.membership

[0, 1, 1, 1, 0, 0, 2, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 2, 0, 1, 1, 2, 3, 0, 2, 2, 2, 2, 1, 1, 0, 2, 1, 0, 1, 0, 0, 1, 1, 2, 2, 0, 1, 2, 2, 1, 2, 2, 2, 1, 0, 0, 0, 0, 0, 1, 1, 0, 2, 1, 0, 3, 2, 0, 1, 2, 2, 1, 2, 0, 0, 2, 2, 1, 2, 0, 0, 0, 0, 0, 0, 1, 1, 0, 2, 2, 0, 2, 0, 1, 2, 1, 0, 0, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 1, 2, 0, 0, 2, 2, 2, 0, 1, 1, 1, 0, 1, 1, 2, 2, 1, 1, 0, 2, 2, 0, 2, 0, 0, 1, 0, 1, 1, 2, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 0, 2, 1, 2, 1, 1, 1, 1, 0, 1, 0, 2, 1, 0, 3, 2, 2, 1, 3, 1, 1, 0, 2, 1, 2, 0, 3, 0, 3, 0, 3, 0, 2, 1, 2, 1, 0, 2, 3, 3, 2, 0, 0, 1, 3, 1, 2, 1, 2, 3, 0, 2, 2, 1, 3, 1, 3, 2, 0, 3, 1, 3, 3, 3, 2, 3, 2, 0, 0, 1, 2, 1, 2, 2, 1, 3, 2, 1, 3, 3, 3, 3, 3, 2, 3, 3, 0, 2, 3, 1, 0, 1, 3, 3, 3, 1, 2, 3, 1, 0, 2, 0, 2, 3, 2]
